# IMPORT LIBRARIES

In [12]:
from optimization_model import build_model
import pyomo.environ as pyo
from __future__ import annotations
import json
import os
import copy


In [13]:
def load_instance_json(path: str) -> Dict[str, Any]:
    """Load instance from JSON. Restores tuple keys for known fields."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Fields with tuple keys and their arity
    tuple_key_fields = {
        "dur": 2,     # (t,k)
        "dist": 2,    # (i,j)
        "tt": 3,      # (i,j,m)
        "Hkp": 2,     # (k,p)
    }

    def decode_keys(d: Dict[str, Any], field_name: str) -> Dict[Any, Any]:
        arity = tuple_key_fields.get(field_name, None)
        if arity is None:
            return d
        out = {}
        for k, v in d.items():
            parts = k.split("|")
            if len(parts) != arity:
                # leave as string key if unexpected
                out[k] = v
            else:
                out[tuple(parts)] = v
        return out

    data: Dict[str, Any] = dict(raw)
    for fn in tuple_key_fields:
        if fn in data and isinstance(data[fn], dict):
            data[fn] = decode_keys(data[fn], fn)

    return data

In [14]:
def export_results(model: pyo.ConcreteModel, filepath: str, save_zero: bool = False):
    """
    Export:
      - objective value
      - all variable values
    to a JSON file.

    save_zero = False -> only store non-zero variables (recommended)
    """

    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)

    results = {}

    # ------------------------
    # Objective
    # ------------------------
    obj = next(model.component_data_objects(pyo.Objective, active=True))
    results["objective_value"] = float(pyo.value(obj))

    # ------------------------
    # Variables
    # ------------------------
    results["variables"] = {}

    for var in model.component_objects(pyo.Var, active=True):
        var_name = var.name
        results["variables"][var_name] = {}

        for index in var:
            val = pyo.value(var[index])
            if (not save_zero) and (abs(val) < 1e-9):
                continue

            # Convert index to string
            if isinstance(index, tuple):
                key = "|".join(map(str, index))
            else:
                key = str(index)

            results["variables"][var_name][key] = float(val)

    # ------------------------
    # Save JSON
    # ------------------------
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {filepath}")

# IMPORT DATA

In [15]:
data = load_instance_json("instances\demo_clustered_seed1.json")
data.keys()

dict_keys(['B', 'K', 'P', 'T', 'D', 'M', 'N', 'a', 'b', 'w', 'dur', 'node', 'base', 'dist', 'tt', 'EF', 'Hkp', 'c_hotel', 'X', 'alpha', 'beta', 'eta', 'gamma', 'c_idle_away', 'delta', '_meta'])

# RUN

In [16]:
# -------------------------
# SCENARIO A: Economic baseline
# -------------------------

data_econ = copy.deepcopy(data)

# Turn off sustainability components
data_econ["beta"] = 0.0   # CO2 weight
data_econ["eta"]  = 0.0   # hotel weight
data_econ["delta"] = 0.0  # idle-away weight

m = build_model(data_econ)
solver = pyo.SolverFactory("cplex")  # or "cbc"/"glpk" depending on your environment
solver.options["timelimit"] = 300   # 5 minuti = 300 secondi
res = solver.solve(m, tee=True)
export_results(m, "results/economic_baseline.json")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpu6pqs8vk.cplex.log' open.
CPLEX> New value for time limit in seconds: 300
CPLEX> Problem 'C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpsx_q_frh.pyomo.lp' read.
Read time = 0.13 sec. (5.62 ticks)
CPLEX> Problem name         : C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpsx_q_frh.pyomo.lp
Objective sense      : Minimize
Variables            :   26900  [Nneg: 20,  Binary: 26880]
Objective nonzeros   :    1210
Linear constraints   :   37687  [Less: 27574,  Greater: 10,  Equal: 10103]
  Nonzeros           :  233326
  RHS nonzeros       :     724

Variables 

In [17]:
# -------------------------
# SCENARIO B: Sustainability-aware
# -------------------------
data_sust = copy.deepcopy(data)

# Turn ON and/or strengthen sustainability components
# You can tune these values to make differences more visible.
data_sust["beta"] = float(data_sust.get("beta", 1.0))     # CO2 weight
data_sust["eta"]  = float(data_sust.get("eta", 1.0))      # hotel weight
data_sust["delta"] = float(data_sust.get("delta", 1.0))   # idle-away weight

m = build_model(data_sust)
solver = pyo.SolverFactory("cplex")  # or "cbc"/"glpk" depending on your environment
solver.options["timelimit"] = 300   # 5 minuti = 300 secondi
res = solver.solve(m, tee=True)
export_results(m, "results/sustainability_aware.json")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpmdkll0no.cplex.log' open.
CPLEX> New value for time limit in seconds: 300
CPLEX> Problem 'C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpjuww2aby.pyomo.lp' read.
Read time = 0.13 sec. (5.65 ticks)
CPLEX> Problem name         : C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpjuww2aby.pyomo.lp
Objective sense      : Minimize
Variables            :   26900  [Nneg: 20,  Binary: 26880]
Objective nonzeros   :   21610
Linear constraints   :   37687  [Less: 27574,  Greater: 10,  Equal: 10103]
  Nonzeros           :  233326
  RHS nonzeros       :     724

Variables 